# BR04 — Resumable five-arm ECG reports

Run once per model in its own working virtual environment/allocation. All models use the same frozen packets and query images. Use a five-report pilot, then MAX_NEW_REPORTS=0. Successes are fsynced and skipped on restart. No simultaneous writers to one model/mode directory. STOP files, signals and run hours stop between reports. Changed prompts, precision, model revision or budgets need a new version; failed deterministic truncation is not silently accepted.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from generation import model_snapshot, load_generator
MODEL_KEY = 'pulse'  # pulse, llama_ecg, llama_base, qwen38
MODEL_PATH = None
DOWNLOAD_MODEL = False  # Optional authenticated Hugging Face download into existing cache.
TOKEN_FILE = '/path/to/private/hf_token'  # First line only; never printed.
ENABLE_THINKING = False  # Only Qwen has this switch; primary experiment is False.
LOAD_IN_4BIT = False
PULSE_REPO = None  # Your working PULSE repository when MODEL_KEY='pulse'.
MODEL_PATH = model_snapshot(MODEL_KEY, MODEL_PATH, download=DOWNLOAD_MODEL,
                            token_file=TOKEN_FILE, pin_dir=RUN / 'model_downloads')
runtime = load_generator(CFG, MODEL_KEY, MODEL_PATH, enable_thinking=ENABLE_THINKING,
                         load_in_4bit=LOAD_IN_4BIT, pulse_repo=PULSE_REPO)
print('Loaded:', runtime['model_id'], runtime['revision'])


In [ ]:
from generation import generate, folder_for
MAX_NEW_REPORTS = 5
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # Less than allocation time remaining AFTER model loading.
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' else 384
# Supplementary thinking comparison: choose the SAME total budget for both Qwen modes.
print(generate(CFG, runtime, max_new=MAX_NEW_REPORTS, retry_errors=RETRY_ERRORS,
               max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS))
FOLDER = folder_for(CFG, MODEL_KEY, ENABLE_THINKING, MAX_NEW_TOKENS)
print('Reports:', FOLDER / 'reports.jsonl')


In [ ]:
from common import latest
from collections import Counter
saved = latest(FOLDER / 'reports.jsonl')
failed = [r for r in saved.values() if r['status'] != 'ok']
print('Failures:', Counter(r.get('error') for r in failed))
for r in list(saved.values())[:5]: print(r['query_id'], r['arm'], r.get('text', r.get('error')))
